In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\heman\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\heman\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import time
import json

START_URL = "https://kathalu.wordpress.com/"
DELAY = 2  

def is_scraping_allowed(url):
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    rp.read()
    return rp.can_fetch("*", url)

def fetch_page(url, delay=DELAY):
    try:
        time.sleep(delay)
        print(f"Fetching: {url}")
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                          'AppleWebKit/537.36 (KHTML, like Gecko) '
                          'Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

def get_internal_links(html_content, base_url):
    soup = BeautifulSoup(html_content, 'html.parser')
    links = set()
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and not href.startswith('#'):
            full_url = urljoin(base_url, href)
            if (urlparse(full_url).netloc == urlparse(base_url).netloc and 
                len(urlparse(full_url).path) > 1 and
                '/tag/' not in full_url and '/category/' not in full_url):
                links.add(full_url)
    return links

def scrape_post_content(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    title_element = soup.find('h1', class_='entry-title') or soup.find('h1', class_='post-title')
    title = title_element.get_text(strip=True) if title_element else "No Title Found"
    content_container = soup.find('div', class_='entry-content') or soup.find('div', class_='post-content')
    main_text = ""
    if content_container:
        paragraphs = content_container.find_all('p')
        main_text = "\n\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))
    return title, main_text


if not is_scraping_allowed(START_URL):
    print("Scraping not allowed by robots.txt. Exiting.")
    exit()

main_html = fetch_page(START_URL)
scraped_data = []
visited = set()

if main_html:
    post_links = get_internal_links(main_html, START_URL)
    print(f"\nFound {len(post_links)} post links. Starting full scrape...")

    for link in post_links:
        if link not in visited:
            visited.add(link)
            post_html = fetch_page(link)
            if post_html:
                title, content = scrape_post_content(post_html)
                scraped_data.append({
                    'url': link,
                    'title': title,
                    'content': content
                })
                print(f" Scraped: {title[:50]}...")


output_file = "kathalu_stories.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=2)

print(f"\nScraping complete. Total stories scraped: {len(scraped_data)}")
print(f"Data saved to {output_file}")


Fetching: https://kathalu.wordpress.com/

Found 61 post links. Starting full scrape...
Fetching: https://kathalu.wordpress.com/2017/05/28/%e0%b0%85%e0%b0%a1%e0%b0%b5%e0%b0%bf%e0%b0%aa%e0%b0%82%e0%b0%a6%e0%b0%bf-%e0%b0%a6%e0%b0%82%e0%b0%a4%e0%b0%be%e0%b0%b2%e0%b1%81/
✅ Scraped: అడవిపంది దంతాలు...
Fetching: https://kathalu.wordpress.com/2021/11/12/%e0%b0%8e%e0%b0%a6%e0%b0%a6-%e0%b0%97%e0%b0%b0%e0%b0%b5/#comments
✅ Scraped: ఎద్దు గర్వం...
Fetching: https://kathalu.wordpress.com/2018/10/27/%e0%b0%aa%e0%b0%bf%e0%b0%9a%e0%b1%81%e0%b0%95-%e0%b0%97%e0%b1%81%e0%b0%a3%e0%b0%82/#comment-1595
✅ Scraped: పిచుక గుణం...
Fetching: https://kathalu.wordpress.com/2017/12/31/%e0%b0%b0%e0%b0%be%e0%b0%9c%e0%b1%81%e0%b0%b2%e0%b1%81-%e0%b0%ae%e0%b0%be%e0%b0%b0%e0%b1%86%e0%b0%a8%e0%b1%8b/
✅ Scraped: రాజులు మారెనో, గుర్రాలు ఎగిరెనో...
Fetching: https://kathalu.wordpress.com/2017/07/04/%e0%b0%95%e0%b0%be%e0%b0%95%e0%b0%bf-%e0%b0%b9%e0%b0%82%e0%b0%b8-%e0%b0%95%e0%b0%be%e0%b0%97%e0%b0%b2%e0%b0%a6%e0%b0%be/
✅ Scrap

In [9]:
pip install selenium beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\heman\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time
import json
import os

BASE_URL = "https://www.manatelugukathalu.com"
CATEGORIES = [  
    "",
    "stories",
    "poems",
    "novels",
    "jokes",
    "articles",
]
DELAY = 3
OUTPUT_JSON = "kathalu_full.json"
OUTPUT_TXT = "kathalu_full.txt"

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

def scroll_to_bottom():
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

def fetch_dynamic_html(url):
    print(f" Fetching: {url}")
    driver.get(url)
    scroll_to_bottom()
    time.sleep(DELAY)
    return driver.page_source

def get_post_links(url):
    html = fetch_dynamic_html(url)
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/post/" in href:
            if not href.startswith("http"):
                href = BASE_URL + href
            links.add(href)
    print(f"Found {len(links)} posts in {url}")
    return list(links)

def scrape_post(url):
    html = fetch_dynamic_html(url)
    soup = BeautifulSoup(html, "html.parser")

    title_meta = soup.find("meta", property="og:title")
    title = title_meta["content"].strip() if title_meta else "No Title"

    paragraphs = soup.find_all("p")
    content = "\n\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))
    return title, content

scraped_data = []
visited = set()

for category in CATEGORIES:
    section_url = f"{BASE_URL}/{category}" if category else BASE_URL
    post_links = get_post_links(section_url)

    for link in post_links:
        if link not in visited:
            visited.add(link)
            try:
                title, content = scrape_post(link)
                if len(content.strip()) < 100:
                    print(f"Skipped empty: {link}")
                    continue

                scraped_data.append({
                    "url": link,
                    "title": title,
                    "content": content,
                    "category": category or "home"
                })
                print(f" Scraped: {title[:70]}...")
            except Exception as e:
                print(f" Error on {link}: {e}")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=2)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    for s in scraped_data:
        f.write(f"{s['title']}\n")
        f.write(s["content"])
        f.write("\n" + "="*80 + "\n")

driver.quit()

print(f"\n Done. Total stories scraped: {len(scraped_data)}")
print(f" JSON saved to: {os.path.abspath(OUTPUT_JSON)}")
print(f" TXT saved to: {os.path.abspath(OUTPUT_TXT)}")
